# Hacer resumenes

## Librerias

In [1]:
import torch
from pathlib import Path
import pandas as pd
import numpy as np
import json

from datasets import Dataset
import evaluate

from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


Torch: 2.7.1
CUDA available: False


## Configuración

In [2]:
CFG = {
    'model_name': "facebook/bart-base",
    'output_dir': "2_Modelos/summarization",
    'max_input_length': 1024,
    'max_target_length': 128,
    'num_epochs': 3,
    'learning_rate': 2e-5,
    'batch_size': 4,
    'gradient_accumulation_steps': 4,
    'warmup_steps': 500,
    'weight_decay': 0.01,
    'save_steps': 500,
    'eval_steps': 500,
    'logging_steps': 100,
}
Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)

GEN_CFG = {
    "max_new_tokens": 60,
    "min_new_tokens": 12,
    "num_beams": 4,
    "early_stopping": True,
    "no_repeat_ngram_size": 3,
    "encoder_no_repeat_ngram_size": 3,   # clave para que no copie del encoder
    "repetition_penalty": 1.2,           # penaliza repetir
    "length_penalty": 0.9,               # <1 = más corto
    "renormalize_logits": True,          # hace el beam más estable
}

print(CFG)
print(GEN_CFG)


{'model_name': 'facebook/bart-base', 'output_dir': '2_Modelos/summarization', 'max_input_length': 1024, 'max_target_length': 128, 'num_epochs': 3, 'learning_rate': 2e-05, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'warmup_steps': 500, 'weight_decay': 0.01, 'save_steps': 500, 'eval_steps': 500, 'logging_steps': 100}
{'max_new_tokens': 60, 'min_new_tokens': 12, 'num_beams': 4, 'early_stopping': True, 'no_repeat_ngram_size': 3, 'encoder_no_repeat_ngram_size': 3, 'repetition_penalty': 1.2, 'length_penalty': 0.9, 'renormalize_logits': True}


## Modelo y token

In [3]:
MODEL_NAME = 'facebook/bart-base'
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model_infer = BartForConditionalGeneration.from_pretrained(MODEL_NAME)  
model_infer.to(DEVICE)


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [4]:
#FT_DIR = Path(CFG['output_dir']) / "final_model"
#tokenizer = BartTokenizer.from_pretrained(FT_DIR)
#model_infer = BartForConditionalGeneration.from_pretrained(FT_DIR)  
#model_infer.to(DEVICE)


In [5]:
def setup_model_and_tokenizer(cfg=CFG):
    tok = BartTokenizer.from_pretrained(cfg['model_name'])
    model = BartForConditionalGeneration.from_pretrained(cfg['model_name'])
    model.to(DEVICE)
    rouge = evaluate.load("rouge")
    return model, tok, rouge

model, tokenizer, rouge = setup_model_and_tokenizer(CFG)


## Carga

In [6]:
def load_splits(csv_path: str, seed: int = 42):
    df = pd.read_csv(csv_path).sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df)
    n_train = int(0.8*n)
    n_val   = int(0.1*n)
    train_df = df.iloc[:n_train]
    val_df   = df.iloc[n_train:n_train+n_val]
    test_df  = df.iloc[n_train+n_val:]
    print(f"Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")
    return train_df, val_df, test_df


## Preprocesamiento y métricas

In [7]:
def preprocess_batch(examples, tokenizer, cfg=CFG):
    X = tokenizer(
        examples['input_text'],
        max_length=cfg['max_input_length'],
        truncation=True,
        padding='max_length'
    )
    Y = tokenizer(
        examples['target_summary'],
        max_length=cfg['max_target_length'],
        truncation=True,
        padding='max_length'
    )
    X['labels'] = Y['input_ids']
    return X

In [8]:
def build_metrics_fn(tokenizer, rouge):
    def _metrics(eval_pred):
        preds, labels = eval_pred
        decoded_preds = []
        for p in preds:
            p_clean = [t for t in p if t is not None and t != -100]
            decoded_preds.append(tokenizer.decode(p_clean, skip_special_tokens=True))
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        res = rouge.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=True
        )
        return {'rouge1': res['rouge1'], 'rouge2': res['rouge2'], 'rougeL': res['rougeL']}
    return _metrics


In [9]:
def make_training_args(cfg):
    from transformers import Seq2SeqTrainingArguments
    import torch
    try:
        return Seq2SeqTrainingArguments(
            output_dir=str(cfg['output_dir']),
            evaluation_strategy="steps",
            eval_steps=cfg['eval_steps'],
            save_strategy="steps",
            save_steps=cfg['save_steps'],
            learning_rate=cfg['learning_rate'],
            per_device_train_batch_size=cfg['batch_size'],
            per_device_eval_batch_size=cfg['batch_size'],
            gradient_accumulation_steps=cfg['gradient_accumulation_steps'],
            num_train_epochs=cfg['num_epochs'],
            warmup_steps=cfg['warmup_steps'],
            weight_decay=cfg['weight_decay'],
            logging_steps=cfg['logging_steps'],
            predict_with_generate=True,
            generation_max_length=cfg['max_target_length'],
            load_best_model_at_end=True,
            metric_for_best_model="rougeL",
            greater_is_better=True,
            save_total_limit=2,
            fp16=False,  
        )
    except TypeError:
        try:
            return Seq2SeqTrainingArguments(
                output_dir=str(cfg['output_dir']),
                eval_strategy="steps",
                eval_steps=cfg['eval_steps'],
                save_steps=cfg['save_steps'],
                learning_rate=cfg['learning_rate'],
                per_device_train_batch_size=cfg['batch_size'],
                per_device_eval_batch_size=cfg['batch_size'],
                gradient_accumulation_steps=cfg['gradient_accumulation_steps'],
                num_train_epochs=cfg['num_epochs'],
                warmup_steps=cfg['warmup_steps'],
                weight_decay=cfg['weight_decay'],
                logging_steps=cfg['logging_steps'],
                predict_with_generate=True,
                generation_max_length=cfg['max_target_length'],
                load_best_model_at_end=True,
                metric_for_best_model="rougeL",
                greater_is_better=True,
                save_total_limit=2,
                fp16=False,
            )
        except TypeError:
            return Seq2SeqTrainingArguments(
                output_dir=str(cfg['output_dir']),
                eval_strategy="steps",
                eval_steps=cfg['eval_steps'],
                save_steps=cfg['save_steps'],
                learning_rate=cfg['learning_rate'],
                per_device_train_batch_size=cfg['batch_size'],
                per_device_eval_batch_size=cfg['batch_size'],
                gradient_accumulation_steps=cfg['gradient_accumulation_steps'],
                num_train_epochs=cfg['num_epochs'],
                warmup_steps=cfg['warmup_steps'],
                weight_decay=cfg['weight_decay'],
                logging_steps=cfg['logging_steps'],
                predict_with_generate=True,
                generation_max_length=cfg['max_target_length'],
                load_best_model_at_end=True,
                save_total_limit=2,
                fp16=False,
            )


## Entrenamiento

In [10]:
def train_model(model, tokenizer, train_df, val_df, cfg=CFG):
    from datasets import Dataset
    from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
    import json
    from pathlib import Path

    print("="*60, "\nINICIANDO FINE-TUNING\n", "="*60, sep='')
    train_ds = Dataset.from_pandas(train_df[['input_text', 'target_summary']])
    val_ds   = Dataset.from_pandas(val_df[['input_text', 'target_summary']])

    train_ds = train_ds.map(
        lambda b: preprocess_batch(b, tokenizer, cfg),
        batched=True,
        remove_columns=['input_text', 'target_summary']
    )
    val_ds = val_ds.map(
        lambda b: preprocess_batch(b, tokenizer, cfg),
        batched=True,
        remove_columns=['input_text', 'target_summary']
    )

    args = make_training_args(cfg)
    collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    metrics_fn = build_metrics_fn(tokenizer, rouge)  

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=metrics_fn,
    )

    train_result = trainer.train()

    out_dir = Path(cfg['output_dir']) / "final_model"
    out_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))

    with open(Path(cfg['output_dir']) / "training_metrics.json", "w") as f:
        json.dump(train_result.metrics, f, indent=2)

    print("\n✓ Fine-tuning completado.")
    return trainer


## Evaluación

In [11]:
def evaluate_model(trainer, tokenizer, test_df, cfg=CFG):
    print("="*60, "\nEVALUACIÓN EN TEST\n", "="*60, sep='')
    test_ds = Dataset.from_pandas(test_df[['input_text', 'target_summary']])
    test_ds = test_ds.map(
        lambda b: preprocess_batch(b, tokenizer, cfg),
        batched=True,
        remove_columns=['input_text', 'target_summary']
    )
    metrics = trainer.evaluate(test_ds)
    for k, v in metrics.items():
        try:
            print(f"{k}: {float(v):.4f}")
        except Exception:
            print(f"{k}: {v}")
    with open(Path(cfg['output_dir']) / "test_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics


In [12]:
# Chunk 1: Función para el Experimento Baseline (CORREGIDA)

def _safe_extract_fmeasure(rouge_score_obj):
    """Extrae el fmeasure de un objeto ROUGE, ya sea float o AggregateScore."""
    # Si tiene el atributo .mid, usa el formato estructurado
    if hasattr(rouge_score_obj, 'mid') and hasattr(rouge_score_obj.mid, 'fmeasure'):
        return rouge_score_obj.mid.fmeasure
    # Si es un float plano (lo que sugiere el error)
    elif isinstance(rouge_score_obj, (float, np.float64)):
        return rouge_score_obj
    return np.nan # Fallback

@torch.inference_mode()
def run_baseline_experiment_summarization(model_name: str, test_df, cfg=CFG, gen_cfg=GEN_CFG, device=DEVICE):
    """
    Evalúa el modelo de resumen pre-entrenado (baseline) generando resúmenes
    en el conjunto de prueba sin realizar fine-tuning y calcula ROUGE.
    """
    print(f"\n{'='*80}\n▶️ Ejecutando Experimento BASELINE: {model_name}\n{'='*80}")
    
    experiment_id = f"{model_name.split('/')[-1].replace('-', '_').upper()}_BASE"
    
    # Cargar tokenizer y modelo base
    try:
        # Nota: BartTokenizer y BartForConditionalGeneration se definieron arriba
        tokenizer = BartTokenizer.from_pretrained(model_name)
        model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
    except Exception as e:
        print(f"❌ Error al cargar el modelo {model_name}: {e}. Omitiendo Baseline.")
        return None
    
    # Preparar DataCollator y DataLoader para el test set
    test_ds = Dataset.from_pandas(test_df[['input_text', 'target_summary']].reset_index(drop=True))
    test_ds = test_ds.map(
        lambda b: preprocess_batch(b, tokenizer, cfg), # Reutiliza tu función preprocess_batch
        batched=True,
        remove_columns=['input_text', 'target_summary']
    )
    
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding='longest')

    test_dataloader = torch.utils.data.DataLoader(
        test_ds, 
        batch_size=cfg['batch_size'], 
        collate_fn=data_collator,
        shuffle=False
    )
    
    metric = evaluate.load("rouge") # Carga la métrica aquí

    generated_summaries = []
    target_summaries = []
    
    model.eval()
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch.get('labels')

        # Generación
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            **gen_cfg # Utiliza la configuración de generación global
        )
        
        # Decodificar
        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        if labels is not None:
            labels_np = np.where(labels.cpu().numpy() != -100, labels.cpu().numpy(), tokenizer.pad_token_id)
            decoded_labels = tokenizer.batch_decode(labels_np, skip_special_tokens=True)
            target_summaries.extend(decoded_labels)

        generated_summaries.extend(decoded_preds)

    # 3. Calcular métricas ROUGE
    metric.add_batch(predictions=generated_summaries, references=target_summaries)
    rouge_scores = metric.compute()
    
    # 4. Formatear resultados usando la función de extracción segura
    results = {
        'experiment_id': experiment_id,
        'model_name': model_name,
        'is_finetuned': False,
        'config': 'BASE',
        'rouge1_fmeasure': _safe_extract_fmeasure(rouge_scores['rouge1']),
        'rouge2_fmeasure': _safe_extract_fmeasure(rouge_scores['rouge2']),
        'rougel_fmeasure': _safe_extract_fmeasure(rouge_scores['rougeL']),
        'generated_summaries': generated_summaries,
        'target_summaries': target_summaries,
    }
    
    print(f"✅ {experiment_id} completado. ROUGE-L F1: {results['rougel_fmeasure']:.4f}")
    
    del model
    torch.cuda.empty_cache()
    
    return results

## Generador de ejemplos

In [13]:
@torch.inference_mode()
def generate_samples_table(model, tokenizer, df, n_samples=5, cfg=CFG, device=DEVICE):
    import pandas as pd
    rows, k = [], min(n_samples, len(df))
    for i in range(k):
        src = str(df.iloc[i]['input_text'])
        tgt = str(df.iloc[i]['target_summary'])
        enc = tokenizer(src, max_length=cfg.get('max_input_length', 512), truncation=True, return_tensors="pt").to(device)

        out = model.generate(
            **enc,
            max_new_tokens=GEN_CFG["max_new_tokens"],
            min_new_tokens=GEN_CFG["min_new_tokens"],
            num_beams=GEN_CFG["num_beams"],
            early_stopping=GEN_CFG["early_stopping"],
            no_repeat_ngram_size=GEN_CFG["no_repeat_ngram_size"],
            encoder_no_repeat_ngram_size=GEN_CFG["encoder_no_repeat_ngram_size"],
            repetition_penalty=GEN_CFG["repetition_penalty"],
            length_penalty=GEN_CFG["length_penalty"],
            renormalize_logits=GEN_CFG["renormalize_logits"],
        )
        gen = tokenizer.decode(out[0], skip_special_tokens=True)

        rows.append({
            "input_full": src,
            "input_preview": (src[:300] + "...") if len(src) > 300 else src,
            "target": tgt,
            "generated": gen,
            "len_input": len(src),
            "len_target": len(tgt),
            "len_generated": len(gen),
        })
    return pd.DataFrame(rows)


## Muestra de uso

In [14]:
#Cargar splits
#train_df, val_df, test_df = load_splits("C:/Users/Alina Tatjana/OneDrive/Documentos/UVG/8vo SEMESTRE/Deep Learning/Proyecto/Proyecto/1_Data/processed/summarization_data.csv")
train_df, val_df, test_df = load_splits("1_Data/processed/summarization_data.csv")

Train=530  Val=66  Test=67


In [ ]:
# Chunk 2: Bucle Principal Unificado (CORREGIDO)

all_results = []
model_name = CFG['model_name']
print(f"\n{'#'*80}\n# Iniciando experimentos para el modelo: {model_name}\n{'#'*80}")

# Cargar splits (el path debe ser local o accesible)
train_df, val_df, test_df = load_splits(
    "1_Data/processed/summarization_data.csv"
)

# 1. 🚀 Ejecutar Experimento BASELINE (Modelo sin Fine-Tuning)
# Esta función ya fue corregida para manejar la extracción de ROUGE de forma robusta.
baseline_result = run_baseline_experiment_summarization(
    model_name, 
    test_df
)
if baseline_result:
    all_results.append(baseline_result)

# 2. 🧠 Ejecutar Experimento de Fine-Tuning
print(f"\n{'='*80}\n▶️ Ejecutando Fine-Tuning\n{'='*80}")

# Entrenar (usando las variables globales model y tokenizer)
trainer = train_model(model, tokenizer, train_df, val_df)
metrics = evaluate_model(trainer, tokenizer, test_df)


################################################################################
# Iniciando experimentos para el modelo: facebook/bart-base
################################################################################
Train=530  Val=66  Test=67

▶️ Ejecutando Experimento BASELINE: facebook/bart-base


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

✅ BART_BASE_BASE completado. ROUGE-L F1: 0.2592

▶️ Ejecutando Fine-Tuning
INICIANDO FINE-TUNING


Map:   0%|          | 0/530 [00:00<?, ? examples/s]

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

...


✓ Fine-tuning completado.
EVALUACIÓN EN TEST


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

/Users/arielamishaancohen/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


eval_loss: 8.3605
eval_rouge1: 0.3802
eval_rouge2: 0.2719
eval_rougeL: 0.3169
eval_runtime: 202.8270
eval_samples_per_second: 0.3300
eval_steps_per_second: 0.0840
epoch: 3.0000


KeyError: 'rouge1'

In [17]:
# Corregimos la extracción: Usamos las claves de métricas reales de la evaluación (sin prefijo).
# Si esto falla, prueba con 'eval_rouge1' o 'test_rouge1'
finetuned_result = {
    'experiment_id': f"{model_name.split('/')[-1].replace('-', '_').upper()}_FT",
    'model_name': model_name,
    'is_finetuned': True,
    'config': 'CONFIG_BASE', 
    'rouge1_fmeasure': metrics['eval_rouge1'], # CORRECCIÓN: Quitamos el prefijo 'test_'
    'rouge2_fmeasure': metrics['eval_rouge2'], # CORRECCIÓN: Quitamos el prefijo 'test_'
    'rougel_fmeasure': metrics['eval_rougeL'], # CORRECCIÓN: Quitamos el prefijo 'test_'
}

all_results.append(finetuned_result)

print("\n" + "="*80)
print("TODOS LOS EXPERIMENTOS COMPLETADOS (Baseline y Fine-Tuning)")
print("="*80)


TODOS LOS EXPERIMENTOS COMPLETADOS (Baseline y Fine-Tuning)


In [18]:
# Chunk 3: Consolidación de Resultados y Comparación (Colocar antes de generate_samples_table)

# Crear DataFrame con resultados comparativos
comparison_data = []

# Iterar sobre la lista all_results
for result in all_results:
    comparison_data.append({
        "Experimento": result["experiment_id"],
        "Modelo Base": result["model_name"],
        "Fine-Tuned": "Sí" if result["is_finetuned"] else "No",
        "ROUGE-1 F1": result["rouge1_fmeasure"],
        "ROUGE-2 F1": result["rouge2_fmeasure"],
        "ROUGE-L F1": result["rougel_fmeasure"],
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values("ROUGE-L F1", ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("🏆 COMPARACIÓN DE MODELOS (BASE vs. FINE-TUNED)")
print("="*80 + "\n")
# Muestra la tabla de comparación
print(comparison_df.to_string(index=False))

# Guardar comparación (opcional)
comparison_df.to_csv(Path(CFG['output_dir']) / "summarization_model_comparison.csv", index=False)
print(f"\n✅ Comparación guardada en: {Path(CFG['output_dir']) / 'summarization_model_comparison.csv'}")


🏆 COMPARACIÓN DE MODELOS (BASE vs. FINE-TUNED)

   Experimento        Modelo Base Fine-Tuned  ROUGE-1 F1  ROUGE-2 F1  ROUGE-L F1
  BART_BASE_FT facebook/bart-base         Sí    0.380214    0.271913    0.316949
BART_BASE_BASE facebook/bart-base         No    0.325572    0.137518    0.259175

✅ Comparación guardada en: 2_Modelos/summarization/summarization_model_comparison.csv


In [19]:
df_samples = generate_samples_table(model_infer, tokenizer, test_df, n_samples=3)
df_samples[["input_preview", "target", "generated", "len_input", "len_target", "len_generated"]]


,input_preview,target,generated,len_input,len_target,len_generated
0,Los Angeles Lakers forward LeBron James 23 pre...,LeBron James injury information leaked to bett...,Los Los Angeles Dodgers forward LeBronJames 23...,1687,238,303
1,"At present, Hims Hers Health generates 159 mil...",Is Hims Hers Health Still a Smart Opportunity ...,"At Present, Him's Hers Health earns 159 millio...",8630,236,253
2,Jamaican officials issued dire warnings Saturd...,Hurricane Melissa takes aim at Jamaica. Jamaic...,JAMAican officials warned dire warningsSaturda...,243,225,244


## Guardar en nuevo CSV

In [21]:
#orig_path = r"C:\Users\Alina Tatjana\OneDrive\Documentos\UVG\8vo SEMESTRE\Deep Learning\Proyecto\Proyecto\1_Data\processed\summarization_data.csv"
orig_path = "1_Data/processed/summarization_data.csv"
df_orig = pd.read_csv(orig_path)

k = len(df_samples)
df_out = df_orig.copy()
df_out.loc[:k-1, "generated"] = df_samples["generated"].values
df_out.loc[:k-1, "target_preview"] = df_samples["target"].values  

out_path = Path(orig_path).with_name(f"summarization_data_with_generated.csv")
df_out.to_csv(out_path, index=False, encoding="utf-8")
print("Guardado en:", out_path)


Guardado en: 1_Data/processed/summarization_data_with_generated.csv
